# Lab 16: TorchScript Export, Latency, Drift, and Capstone Review

            **Duration:** 3 hours  
            **Lecture alignment:** Week 16 — Deployment, monitoring, comparative evaluation, and project review  
            **CLO mapping:** CLO-2, CLO-3, CLO-4, CLO-5  
            **Framework:** PyTorch (standalone, credential-free, CPU smoke-test with optional GPU)

            ## Learning objectives

            - Run a reproducible paired model comparison with quality and cost metrics.
- Export and verify a TorchScript artifact without a live server.
- Measure latency and simulated drift, then package a responsible result card.

            ## Three-hour activity plan

            - 0–35 min: clean-run and paired experiment
- 35–75 min: quality, parameters, and latency comparison
- 75–110 min: TorchScript export/equivalence
- 110–145 min: drift/monitoring analysis
- 145–180 min: result card, demonstration, peer review, and course reflection


## Book grounding

            - Huyen, *Designing Machine Learning Systems*, O'Reilly Media, 2022.
- Bishop and Bishop, *Deep Learning: Foundations and Concepts*, Springer, 2024.
- Zhang, Lipton, Li, and Smola, *Dive into Deep Learning*, Cambridge University Press, 2024.

            The notebook paraphrases concepts and supplies original code; it does not reproduce book text.


In [ ]:
from pathlib import Path
import json, math, os, random, time
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import DataLoader, TensorDataset

FAST_MODE = True
RUN_EXTENSION = False
SEED = 20276
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_num_threads(min(2, os.cpu_count() or 1))

if Path("/content").exists():
    ARTIFACT_DIR = Path("/content/artifacts/lab_16")
else:
    ARTIFACT_DIR = Path.cwd() / "tmp" / "course_build" / "runtime" / "lab_16"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print({"lab": 16, "device": str(DEVICE), "fast_mode": FAST_MODE,
       "artifacts": str(ARTIFACT_DIR), "torch": torch.__version__})


## Predict before running

Assess whether the nonlinear MLP will justify its extra parameters and latency, predict how the simulated feature shift will alter confidence, and define an alert threshold.

Record a brief prediction in your own words before executing the experiment, then revisit it in the exit reflection.


## Activity 1 — Reproducible paired capstone experiment


In [ ]:
n=1100 if FAST_MODE else 5000;X=torch.randn(n,6);score=1.2*X[:,0]-.9*X[:,1]+.7*X[:,2]*X[:,3]+.25*torch.randn(n);y=(score>0).long()
split=int(.75*n);Xtr,Xte,ytr,yte=X[:split],X[split:],y[:split],y[split:]
models={"linear":nn.Linear(6,2),"mlp":nn.Sequential(nn.Linear(6,24),nn.ReLU(),nn.Linear(24,12),nn.ReLU(),nn.Linear(12,2))}
def fit(name,model):
    torch.manual_seed(SEED);model=model.to(DEVICE);opt=torch.optim.Adam(model.parameters(),lr=.02);history=[]
    for _ in range(40 if FAST_MODE else 120):
        opt.zero_grad();loss=F.cross_entropy(model(Xtr.to(DEVICE)),ytr.to(DEVICE));loss.backward();opt.step();history.append(loss.item())
    model.eval();
    with torch.no_grad():logits=model(Xte.to(DEVICE));pred=logits.argmax(1).cpu();probs=logits.softmax(1)[:,1].cpu()
    tp=((pred==1)&(yte==1)).sum().item();fp=((pred==1)&(yte==0)).sum().item();fn=((pred==0)&(yte==1)).sum().item()
    precision=tp/(tp+fp+1e-8);recall=tp/(tp+fn+1e-8);f1=2*precision*recall/(precision+recall+1e-8)
    start=time.perf_counter()
    with torch.no_grad():
        for _ in range(100):_ = model(Xte.to(DEVICE))
    latency=(time.perf_counter()-start)/(100*len(Xte))
    return model,history,{"accuracy":(pred==yte).float().mean().item(),"f1":f1,"parameters":sum(p.numel() for p in model.parameters()),"seconds_per_item":latency},pred,probs
results={name:fit(name,model) for name,model in models.items()};summary={name:value[2] for name,value in results.items()};print(json.dumps(summary,indent=2))
best_name=max(summary,key=lambda name:summary[name]["f1"]);best_model=results[best_name][0]


## Activity 2 — TorchScript export and equivalence


In [ ]:
best_model.eval();example=Xte[:8].to(DEVICE);scripted=torch.jit.trace(best_model,example);script_path=ARTIFACT_DIR/"capstone_model.ts";scripted.save(str(script_path))
loaded=torch.jit.load(str(script_path),map_location=DEVICE)
with torch.no_grad():native_output=best_model(example);script_output=loaded(example)
max_export_error=(native_output-script_output).abs().max().item()
torch.save({"model":best_model.state_dict(),"model_name":best_name,"seed":SEED,"features":6},ARTIFACT_DIR/"capstone_checkpoint.pt")
print({"best_model":best_name,"max_export_error":max_export_error,"torchscript_bytes":script_path.stat().st_size})


## Activity 3 — Latency, drift, robustness, and capstone result card


In [ ]:
shifted=Xte.clone();shifted[:,0]+=1.25;shifted[:,4]*=1.8
with torch.no_grad():clean_conf=best_model(Xte.to(DEVICE)).softmax(1).max(1).values.cpu();shift_conf=best_model(shifted.to(DEVICE)).softmax(1).max(1).values.cpu()
train_mean=Xtr.mean(0);train_std=Xtr.std(0).clamp_min(1e-6);drift_vector=((shifted.mean(0)-train_mean)/train_std).abs();drift_score=drift_vector.max().item()
fig,axes=plt.subplots(1,2,figsize=(10,3.6))
for name,value in results.items():axes[0].plot(value[1],label=name)
axes[0].legend();axes[0].set(title="Capstone training loss",xlabel="epoch")
axes[1].hist(clean_conf,bins=15,alpha=.6,label="clean");axes[1].hist(shift_conf,bins=15,alpha=.6,label="shifted");axes[1].legend();axes[1].set_title(f"Confidence under drift (score={drift_score:.2f})")
fig.tight_layout();fig.savefig(ARTIFACT_DIR/"capstone_monitoring.png",dpi=150);plt.show()
result_card={"task":"synthetic binary classification","best_model":best_name,"comparison":summary,"export_max_abs_error":max_export_error,
             "drift_score_max_standardized_mean":drift_score,"clean_mean_confidence":clean_conf.mean().item(),"shifted_mean_confidence":shift_conf.mean().item(),
             "limitations":["synthetic data","drift score is illustrative","no subgroup attribute is present, so no fairness claim is supported"]}
(ARTIFACT_DIR/"RESULT_CARD.json").write_text(json.dumps(result_card,indent=2));print(json.dumps(result_card,indent=2))


## Automated checks


In [ ]:
assert max_export_error<1e-5 and script_path.exists()
assert all(value[2]["accuracy"]>.70 for value in results.values())
assert drift_score>1 and torch.isfinite(clean_conf).all() and torch.isfinite(shift_conf).all()
assert (ARTIFACT_DIR/"RESULT_CARD.json").exists() and (ARTIFACT_DIR/"capstone_monitoring.png").exists()
print("All Lab 16 checks passed.")


## Deliverables

                - Restart-and-run-all paired experiment
- TorchScript model and equivalence evidence
- Latency/drift visualization
- Result card with explicit limitations and capstone recommendation

                Submit the executed notebook and the files created in `/content/artifacts/lab_16/`.


## Disabled extension

The following challenge is intentionally disabled by default so the CPU baseline stays quick.


In [ ]:
if RUN_EXTENSION:
    quantized=torch.quantization.quantize_dynamic(best_model,{nn.Linear},dtype=torch.qint8)
    print("Dynamic quantized parameter representation created:",quantized)
else:
    print("Extension disabled: dynamic quantization or a task-specific robustness test.")


## Exit reflection

In 4–6 sentences, state: (1) whether your prediction was supported, (2) the strongest evidence,
(3) one failure mode or limitation, and (4) the next experiment you would run. Include at least
one measured value rather than only a general claim.
